In [9]:
data = {
  "_id": "1-889fd444-e543-47f2-9104-a94504b3e888",
  "player_ids": [
    "8f411492-8a71-4bee-a42d-30961aa448f2",
    "eeac2a93-a74f-4922-999d-e9e3ac37bbad",
    "0c2729dc-6759-4a9a-9884-ef1fac446a7d",
    "f364b3d5-8ee1-465c-92f8-a3fcfa13c05b",
    "535b1d8b-1800-44fa-9f8a-85a67a083194",
    "61fad0c5-473e-4dfe-aba1-1dab3afe4d72",
    "245f7052-b832-47b9-afff-867716ece487",
    "374ed655-fbd2-450a-83e3-de15eba368b1",
    "72b3947d-e493-4638-b812-58169e9a4332",
    "b8848d1b-4bc3-4521-bbf5-eeeca74e3df6"
  ],
  "timestamp": {
    "started_at": 1788214167,
    "finished_at": 1788216033
  }
}

In [10]:
import polars as pl
from pipeline.orch import getdata

client = getdata()
client.connect_db(
    'localhost', 27017
)

data = list(client.alters.find({}))


In [11]:
df = pl.DataFrame(data).explode('player_ids').unnest('timestamp')



/tmp/ipykernel_6749/4065549699.py:1: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  df = pl.DataFrame(data).explode('player_ids').unnest('timestamp')


In [12]:
agg_df = df.group_by("player_ids").agg(
    pl.col('_id').alias('match_ids'),
    pl.col('started_at').max().alias('max_started_at'),
    (pl.col('started_at').min().alias('min_started_at') - 2592000)
)

agg_df.to_dicts()


[{'player_ids': 'bdf5e42d-fe7f-46c8-85cb-aa8ed7dfa174',
  'match_ids': ['1-97049e62-c2e7-4673-8508-e82ce3be6731'],
  'max_started_at': 1785884341,
  'min_started_at': 1783292341},
 {'player_ids': 'dec0b03f-3ec0-4d18-a2d8-bc46a7781946',
  'match_ids': ['1-128833e8-85ca-464a-8d78-c1b58c070dbd'],
  'max_started_at': 1788473544,
  'min_started_at': 1785881544},
 {'player_ids': 'ac8013d7-79fc-4887-9dc9-4dfe3ae504b7',
  'match_ids': ['1-8311ff50-30a3-4871-84df-f38b7779832b'],
  'max_started_at': 1786307814,
  'min_started_at': 1783715814},
 {'player_ids': '412cea42-9afb-432d-aabb-3f1651fe231b',
  'match_ids': ['1-d1f949a2-f9cc-4a44-a75d-ea05eadea78b'],
  'max_started_at': 1788277203,
  'min_started_at': 1785685203},
 {'player_ids': '74fd8d8c-1863-40a6-94de-a8a0d37fe1c5',
  'match_ids': ['1-7d3e0753-3e1c-4e09-a533-0c83768301f9',
   '1-9ea6f325-a9b9-4105-b97e-0e0861fa4960',
   '1-548f3bf4-d213-42a1-97c9-1ad99b224c1e',
   '1-0e42b27d-5e88-4a4c-851a-b1de74a7c61b'],
  'max_started_at': 1788804796

In [13]:
agg_df.to_dicts()

[{'player_ids': 'bdf5e42d-fe7f-46c8-85cb-aa8ed7dfa174',
  'match_ids': ['1-97049e62-c2e7-4673-8508-e82ce3be6731'],
  'max_started_at': 1785884341,
  'min_started_at': 1783292341},
 {'player_ids': 'dec0b03f-3ec0-4d18-a2d8-bc46a7781946',
  'match_ids': ['1-128833e8-85ca-464a-8d78-c1b58c070dbd'],
  'max_started_at': 1788473544,
  'min_started_at': 1785881544},
 {'player_ids': 'ac8013d7-79fc-4887-9dc9-4dfe3ae504b7',
  'match_ids': ['1-8311ff50-30a3-4871-84df-f38b7779832b'],
  'max_started_at': 1786307814,
  'min_started_at': 1783715814},
 {'player_ids': '412cea42-9afb-432d-aabb-3f1651fe231b',
  'match_ids': ['1-d1f949a2-f9cc-4a44-a75d-ea05eadea78b'],
  'max_started_at': 1788277203,
  'min_started_at': 1785685203},
 {'player_ids': '74fd8d8c-1863-40a6-94de-a8a0d37fe1c5',
  'match_ids': ['1-7d3e0753-3e1c-4e09-a533-0c83768301f9',
   '1-9ea6f325-a9b9-4105-b97e-0e0861fa4960',
   '1-548f3bf4-d213-42a1-97c9-1ad99b224c1e',
   '1-0e42b27d-5e88-4a4c-851a-b1de74a7c61b'],
  'max_started_at': 1788804796

In [14]:
list1= agg_df.select(
    "player_ids",
    pl.col("match_ids").list.len().alias("n_matches"),
).sort("n_matches", descending=True)['player_ids'][50:100]

import requests
import os
from dotenv import load_dotenv

load_dotenv()


API_KEY = os.environ['API_KEY']

for PLAYER_ID in list1:
    #PLAYER_ID = "75065f7e-4cac-4a39-9e0d-617a0293304c"

    url = f"https://open.faceit.com/data/v4/players/{PLAYER_ID}"

    headers = {
        "Authorization": f"Bearer {API_KEY}"
    }

    response = requests.get(url, headers=headers)
    response.raise_for_status()

    player = response.json()

    print("FACEIT name:", player["nickname"])

FACEIT name: Muzz_1
FACEIT name: -Pops1clE-
FACEIT name: Nikoinfinity
FACEIT name: gunsnatcher
FACEIT name: -Neme
FACEIT name: RUSHHHH-
FACEIT name: -Swa_
FACEIT name: Migrane
FACEIT name: 6yar
FACEIT name: -WARDELL
FACEIT name: Jimbo-xD
FACEIT name: Hunaid--
FACEIT name: SKIZZY47
FACEIT name: -Odachi
FACEIT name: rop-
FACEIT name: MoJisuke
FACEIT name: C1S-
FACEIT name: N0INF0
FACEIT name: -ArchNemesis
FACEIT name: Jaanooo1
FACEIT name: 1MMMMMMMMMMM
FACEIT name: ooooooooook
FACEIT name: Pashaaaa-
FACEIT name: ELPUNISHER
FACEIT name: VARLIX
FACEIT name: -Rabb1tz
FACEIT name: ---Rose-
FACEIT name: Lone-Wol7
FACEIT name: -_koi
FACEIT name: -Kayxee-
FACEIT name: --_Jerry_--
FACEIT name: -Tri
FACEIT name: ammaRRRRRRR
FACEIT name: 4rslan-
FACEIT name: ClassicMan_
FACEIT name: DatuMaro
FACEIT name: -420W1CK3D
FACEIT name: R1p999
FACEIT name: iBaaaaaaad
FACEIT name: adnanNn1
FACEIT name: rAiin-
FACEIT name: -_electro-_-
FACEIT name: -protagon1st
FACEIT name: AllOver-
FACEIT name: --BENN--
FAC

In [15]:
agg_df.select(
    "player_ids",
    pl.col("match_ids").list.len().alias("n_matches"),
).sort("n_matches", descending=True)[50:100]

player_ids,n_matches
str,u32
"""ce9bb281-3b2d-4ca2-9a26-9b1e0d…",85
"""5b5bbaa5-2eca-476c-ad17-0dbe50…",85
"""1a046637-fe13-400c-839f-150adc…",85
"""64833aaf-85c3-459b-98cd-7d8d58…",85
"""225fc6b4-91a4-454b-9dd8-a39ac4…",83
…,…
"""cbaeaa1d-7b4a-4cc4-8d28-3c1b56…",74
"""093ba79a-8e2a-402a-bb3b-74ae3e…",74
"""c789348f-eaee-47f8-848d-48ed9d…",73


In [16]:
data = list(client.alters.find({}))
df = pl.DataFrame(data).explode('player_ids').unnest('timestamp')

agg_df = df.group_by("player_ids").agg(
    pl.struct(
        pl.col("_id"),
        pl.col("started_at")
    ).alias("match_ids"),
    pl.col('started_at').max().alias('max_started_at'),
    (pl.col('started_at').min().alias('min_started_at') - 2592000)
)

final_df = agg_df.to_dicts()
final_df[:1]

/tmp/ipykernel_6749/3983081841.py:2: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  df = pl.DataFrame(data).explode('player_ids').unnest('timestamp')


[{'player_ids': 'fdc2fc1f-b2c5-4b5f-baa9-6170ac9b4362',
  'match_ids': [{'_id': '1-fdd05174-49e2-4712-9a84-2542ad1d4599',
    'started_at': 1787605801}],
  'max_started_at': 1787605801,
  'min_started_at': 1785013801}]

In [17]:
from pympler import asizeof

data_mb = asizeof.asizeof(data) / (1024 ** 2)
final_df_mb = asizeof.asizeof(final_df) / (1024 ** 2)

print(f"data:     {data_mb:.2f} MB")
print(f"final_df: {final_df_mb:.2f} MB")

data:     50.68 MB
final_df: 172.38 MB


In [19]:
import polars as pl
import requests
import os
from dotenv import load_dotenv
from datetime import datetime

load_dotenv()
API_KEY = os.environ["API_KEY"]

top_player = (
    agg_df
    .with_columns(
        pl.col("match_ids").list.len().alias("n_matches")
    )
    .sort("n_matches", descending=True)
    .row(0, named=True)
)

PLAYER_ID = top_player["player_ids"]
MIN_STARTED_AT = top_player["min_started_at"]
MAX_STARTED_AT = top_player["max_started_at"]

url = f"https://open.faceit.com/data/v4/players/{PLAYER_ID}/history"
headers = {"Authorization": f"Bearer {API_KEY}"}

all_matches = []
limit = 100
offset = 0

while offset <= 1000:
    params = {
        "game": "cs2",
        "from": MIN_STARTED_AT,
        "to": MAX_STARTED_AT,
        "limit": limit,
        "offset": offset,
    }

    r = requests.get(url, headers=headers, params=params)
    r.raise_for_status()

    matches = r.json().get("items", [])
    all_matches.extend(matches)

    print(f"offset={offset} -> {len(matches)} matches")

    if len(matches) < limit:
        break

    offset += limit

print("\nPlayer:", PLAYER_ID)
print("Matches in agg_df:", top_player["n_matches"])
print("Min start:", MIN_STARTED_AT, datetime.fromtimestamp(MIN_STARTED_AT))
print("Max start:", MAX_STARTED_AT, datetime.fromtimestamp(MAX_STARTED_AT))
print("Total FACEIT matches returned:", len(all_matches))
print("Unique matches:", len({m["match_id"] for m in all_matches}))

offset=0 -> 100 matches
offset=100 -> 100 matches
offset=200 -> 100 matches
offset=300 -> 100 matches
offset=400 -> 100 matches
offset=500 -> 100 matches
offset=600 -> 100 matches
offset=700 -> 100 matches
offset=800 -> 64 matches

Player: 75065f7e-4cac-4a39-9e0d-617a0293304c
Matches in agg_df: 153
Min start: 1779671085 2026-05-25 05:04:45
Max start: 1789504531 2026-09-16 00:35:31
Total FACEIT matches returned: 864
Unique matches: 864


In [20]:
all_matches

[{'match_id': '1-4f52c127-1de4-408c-a4b6-b48286356f01',
  'game_id': 'cs2',
  'region': 'EU',
  'match_type': '',
  'game_mode': '5v5',
  'max_players': 10,
  'teams_size': 5,
  'teams': {'faction1': {'team_id': 'd9d1a97e-56f4-466c-a272-874e5f2ae369',
    'nickname': 'team_C4pr1corn',
    'avatar': 'https://assets.faceit-cdn.net/avatars/4f5d5e30-7b77-40f7-a655-2094e9ddc999_1551272294700.jpg',
    'type': '',
    'players': [{'player_id': '4f5d5e30-7b77-40f7-a655-2094e9ddc999',
      'nickname': 'C4pr1corn',
      'avatar': 'https://assets.faceit-cdn.net/avatars/4f5d5e30-7b77-40f7-a655-2094e9ddc999_1551272294700.jpg',
      'skill_level': 8,
      'game_player_id': '76561198116406598',
      'game_player_name': 'C4pr1c0rn (Mid)',
      'faceit_url': 'https://www.faceit.com/{lang}/players/C4pr1corn'},
     {'player_id': 'e47f3329-3033-4a11-b43d-7d0ca506518a',
      'nickname': '9999999_',
      'avatar': 'https://distribution.faceit-cdn.net/images/d9fca1e0-c7b9-4fbb-aa36-46b3599ab674.jpg

In [21]:
all_matches_mb = asizeof.asizeof(all_matches) / (1024 ** 2)
print(f"final_df: {all_matches_mb:.2f} MB")

final_df: 9.27 MB


In [22]:
import requests

session = requests.Session()
session.headers.update({
    "Authorization": f"Bearer {API_KEY}",
    "Accept": "application/json"
})

stats_data = []

for i, match in enumerate(all_matches, 1):
    match_id = match["match_id"]
    started_at = match["started_at"]

    url = f"https://open.faceit.com/data/v4/matches/{match_id}/stats"
    r = session.get(url)

    if r.status_code in (403, 404):
        stats_data.append({
            "_id": match_id,
            "started_at": started_at,
            "stats_missing": True,
            "status_code": r.status_code,
            "stats": None
        })
        print(f"{i}/{len(all_matches)} | {r.status_code} skipped", end="\r")
        continue

    r.raise_for_status()

    stats_data.append({
        "_id": match_id,
        "started_at": started_at,
        "stats_missing": False,
        "status_code": 200,
        "stats": r.json()
    })

    print(f"{i}/{len(all_matches)} | fetched", end="\r")

print("\nDone")
print("200:", sum(x["status_code"] == 200 for x in stats_data))
print("403:", sum(x["status_code"] == 403 for x in stats_data))
print("404:", sum(x["status_code"] == 404 for x in stats_data))


864/864 | fetchedped
Done
200: 863
403: 0
404: 1


In [33]:
all_used_ids = set()

for row in aggregated.to_dicts():
    all_used_ids.update(row["match_ids_15d"])
    all_used_ids.update(row["match_ids_30d"])

print("Unique matches used:", len(all_used_ids))
print("Original stats_data:", len(stats_data))
print("Difference:", len(stats_data) - len(all_used_ids))

Unique matches used: 181
Original stats_data: 864
Difference: 683


In [34]:
used_ids = set()

for row in aggregated.to_dicts():
    used_ids.update(row["match_ids_15d"])
    used_ids.update(row["match_ids_30d"])

unused = stats_df.filter(
    ~pl.col("_id").is_in(list(used_ids))
)

print(unused.select("_id", "started_at").sort("started_at"))

shape: (682, 2)
┌─────────────────────────────────┬────────────┐
│ _id                             ┆ started_at │
│ ---                             ┆ ---        │
│ str                             ┆ i64        │
╞═════════════════════════════════╪════════════╡
│ 1-394c95a9-4ad5-41cf-ba0e-5764… ┆ 1780872365 │
│ 1-a494b23b-83cc-4144-9b94-9afa… ┆ 1780875854 │
│ 1-3fb2b4a3-546e-4e9c-88fa-9902… ┆ 1780877446 │
│ 1-8c41fc72-a07a-49de-8c43-b75f… ┆ 1780880535 │
│ 1-b41216a4-10d5-43e3-af39-98bc… ┆ 1780954763 │
│ …                               ┆ …          │
│ 1-5b3ed2ba-a522-47b8-ad40-7da8… ┆ 1789497221 │
│ 1-f68c4de6-8b45-478f-9160-3ad6… ┆ 1789499601 │
│ 1-d647c6e1-cb1e-4602-9aca-6fbc… ┆ 1789501859 │
│ 1-feaa8570-aed9-48ed-a4e8-5a40… ┆ 1789504531 │
│ 1-4f52c127-1de4-408c-a4b6-b482… ┆ 1789506830 │
└─────────────────────────────────┴────────────┘


In [35]:
aggregate_statistics(matches_fetch_df, stats_df)

_id,started_at,fifteen_window,thirty_window,match_ids_15d,match_ids_30d
str,i64,i64,i64,list[str],list[str]
"""1-d98000ad-ef4a-460a-ac15-ca45…",1789477665,1788181665,1786885665,"[""1-623c3411-1962-455f-ae79-773b4f229c4a"", ""1-82c9d0cb-3deb-46b3-a50f-a3b4c954c17a"", … ""1-71ab8bfc-c30d-4084-b032-ba4e2ca6cfae""]","[""1-96362b0e-ae99-4457-ba80-ed54393341dd"", ""1-9f94968b-aa49-495e-b80e-f96d6fcf243c"", … ""1-ecbbeeb0-4556-4ea2-b7a7-28e9602886ac""]"


In [ ]:
aggregated_df = aggregate_statistics(


    , stats_df).to_dicts()

stats_lookup = dict(
    zip(
        stats_df["_id"].to_list(),
        stats_df["started_at"].to_list()
    )
)

for row in aggregated_df:

    anchor_time = row["started_at"]
    fifteen = row["fifteen_window"]
    thirty = row["thirty_window"]

    actual_15 = set(row["match_ids_15d"])
    actual_30 = set(row["match_ids_30d"])

    expected_15 = {
        match_id
        for match_id, t in stats_lookup.items()
        if fifteen <= t < anchor_time
    }

    expected_30 = {
        match_id
        for match_id, t in stats_lookup.items()
        if thirty <= t < fifteen
    }

    assert actual_15 == expected_15, f"15d WRONG for {row['_id']}"
    assert actual_30 == expected_30, f"30d WRONG for {row['_id']}"
    assert actual_15.isdisjoint(actual_30), f"OVERLAP for {row['_id']}"

print("All joins/windows are correct.")

SyntaxError: invalid syntax (3495522423.py, line 4)